# Advection et conduction dans une colonne nappe-riviere (M2)

Objectifs : comprendre comment un gradient de charge (infiltration ou exfiltration) transporte la chaleur, et comment la porosite, la conductivite thermique et la permeabilite controlent le signal de temperature mesure a quelques decimetres sous le lit.

## 1. Equations

Transport d'energie dans le milieu poreux sature (une dimension, axe $z$ oriente vers le haut) :

$$
\underbrace{\left[\phi C_w + (1-\phi) C_s\right]}_{C_{v}} \frac{\partial T}{\partial t}
= \underbrace{\lambda \frac{\partial^2 T}{\partial z^2}}_{\text{conduction}}
- \underbrace{\phi C_w\, q \frac{\partial T}{\partial z}}_{\text{advection}}
$$

- $\phi$ : porosite, $C_w$ : capacite volumique de l'eau, $C_s = \rho_s c_s$ : capacite volumique du solide [J m$^{-3}$ K$^{-1}$] (masse volumique $\times$ capacite specifique).
- $\lambda$ : conductivite thermique du milieu [W m$^{-1}$ K$^{-1}$].
- $q$ : vitesse de Darcy [m/s], positive vers le haut (exfiltration) dans la convention Ginette.

Loi de Darcy, avec la charge $h$ :

$$
q = -\frac{k \rho g}{\mu} \frac{\partial h}{\partial z}
\quad\Rightarrow\quad
q \approx -K \frac{h_{\text{top}} - h_{\text{bottom}}}{L}
$$

**Attention au signe.** Si $h_{\text{top}} > h_{\text{bottom}}$, l'eau descend : c'est une **infiltration** (riviere vers nappe). Si $h_{\text{top}} < h_{\text{bottom}}$, l'eau monte : c'est une **exfiltration** (nappe vers riviere).

## 2. Nombre de Peclet

Le rapport entre advection et conduction sur la longueur $L$ de la colonne :

$$
\mathrm{Pe} = \frac{q\, C_w\, L}{\lambda}
$$

- $|\mathrm{Pe}| \ll 1$ : la conduction domine, le profil de temperature est celui d'un milieu immobile.
- $|\mathrm{Pe}| \gg 1$ : l'advection domine, la chaleur est transportee par l'eau.

Avant de simuler, calculez le nombre de Peclet pour les trois cas du TD.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(Path.cwd().resolve().parents[1] / 'src' / 'src_python'))
from Init_folders import setup_lab_environment
from Direct_model import (setup_ginette, initial_conditions, boundary_conditions,
                          run_direct_model, write_coordonnee_file, remove_first_days_sim)
td_dir, synth_dir = setup_lab_environment()
print('Dossier de travail :', synth_dir)


Current working directory: /Users/ariviere/Programmes/ginette/application/2026_TD_ENS/SYNTHETIC_CASES
ginette exists
Dossier de travail : /Users/ariviere/Programmes/ginette/application/2026_TD_ENS/SYNTHETIC_CASES


## 3. Geometrie et constantes

Colonne de 5 m, maille de 1 cm (modifiable : `dz` doit diviser 5 m). Les capteurs sont aux centres de maille : T1 a 10,5 cm, T2 a 20,5 cm, T3 a 30,5 cm, T4 a 40,5 cm et T5 a 100,5 cm sous la surface.

In [2]:
z_top, z_bottom = 0.0, -5.0
az = z_top - z_bottom
dz = 0.01
n_depths = round(az / dz)
assert abs(n_depths * dz - az) < 1e-9, 'dz doit diviser la hauteur de colonne'
dz_obs = 0.1
dt = 600
nb_day = 30          # duree analysee (jours), hors spin-up
T_SPIN = 10          # jours de mise en route (spin-up) : simules puis retires des resultats
state = 1
date_simul_bg = pd.to_datetime('2022/04/21 14:00:00')
mu, rho, g = 1e-3, 1000.0, 9.81
Cw_vol = 4.185e6   # J/m3/K
print(f'{n_depths} mailles de {dz} m, capteurs a {dz_obs} m d\'intervalle')


500 mailles de 0.01 m, capteurs a 0.1 m d'intervalle


## 4. Un cas = un jeu de parametres

La fonction ci-dessous construit les conditions aux limites pour un regime donne et lance Ginette. Le regime est fixe par la difference de charge moyenne `dh` (m) entre le haut et le bas de la colonne :
- `dh = 0` : conduction pure (controle)
- `dh > 0` : infiltration
- `dh < 0` : exfiltration

La temperature de surface a un cycle journalier (amplitude 5 °C, periode 1 jour) autour de 17 °C ; la base est maintenue a 10 °C (temperature imposee).

In [3]:
PROFIL_JOURS = [2, 5, 10, 20, 29]   # dates (jours depuis la fin du spin-up) ou l'on extrait les profils
# capteurs T1 a T5 : centres de maille (dz = 1 cm), cotes en m
capteurs = {'T10.5': -0.105, 'T20.5': -0.205, 'T30.5': -0.305, 'T40.5': -0.405, 'T100.5': -1.005}
t_spin = T_SPIN * 86400   # duree du spin-up [s]

def retirer_spin_up(d):
    # remove_first_days_sim (src/src_python/Direct_model.py) retire les jours du spin-up ;
    # ensuite le temps repart de 0 au debut de l'analyse
    d = remove_first_days_sim(d, T_SPIN).copy()
    d['Time'] = d['Time'] - t_spin
    return d.reset_index(drop=True)

def simuler(dh=0.0, n=0.4, log_k=-10.8, lam=1.0, c_solid=800.0, nb_jours=nb_day):
    # la simulation dure spin-up + nb_jours ; le spin-up est retire des resultats
    nb_total = T_SPIN + nb_jours
    t = np.arange(0, nb_total * 86400, dt)
    df = pd.DataFrame({'times': t, 'days': t / 86400})
    df['T_top'] = 5 * np.sin(2 * np.pi * df['days']) + 17
    df['T_bottom'] = 10.0
    df['h_bottom'] = 5.0
    df['h_top'] = 5.0 + dh
    df['head_gradient'] = dh / az
    k = 10 ** log_k
    K = k * rho * g / mu
    z_obs = setup_ginette(dt, state, nb_total, z_top, z_bottom, az, dz, date_simul_bg, dz_obs)
    write_coordonnee_file(z_bottom, dz)
    initial_conditions(df, z_top, z_bottom, dz, z_obs)
    boundary_conditions(df, dt)
    sim = run_direct_model(date_simul_bg, z_bottom, dz, 1, 0.0, log_k, n, lam, c_solid)
    flux_eau = pd.read_csv('Sim_velocity_profil_t.dat', sep=r'\s+', header=None, names=['Time', 'z', 'q'])
    flux_chal = pd.read_csv('Sim_heat_flux_profil_t.dat', sep=r'\s+', header=None,
                            names=['Time', 'z', 'Conductive', 'Advective', 'Total'])
    # profils de temperature (toutes les mailles) aux dates PROFIL_JOURS, comptees depuis la fin du spin-up
    # le fichier est ecrase a chaque run
    profil = pd.read_csv('Sim_temperature_profil_t.dat', sep=r'\s+', header=None, names=['Time', 'z', 'T'])
    a_garder = np.isclose(profil['Time'].values[:, None], t_spin + np.array(PROFIL_JOURS) * 86400).any(axis=1)
    # temperatures aux capteurs : centres de maille (profondeur exacte, pas d'interpolation)
    P = profil.pivot(index='Time', columns='z', values='T')
    for nom, zc_capt in capteurs.items():
        maille = P.columns[np.argmin(np.abs(P.columns.values - zc_capt))]
        sim[nom] = P[maille].reindex(sim['Time'].values).values
    sim = retirer_spin_up(sim)
    sim.attrs['flux_eau'] = retirer_spin_up(flux_eau)
    sim.attrs['flux_chal'] = retirer_spin_up(flux_chal)
    sim.attrs['profils'] = retirer_spin_up(profil[a_garder])
    q = -K * dh / az            # vitesse de Darcy [m/s], positive vers le haut
    Pe = q * Cw_vol * az / lam
    return sim, q, Pe
print('fonction simuler prete')


fonction simuler prete


### Schema du domaine

La colonne modelisee (5 m, mailles de 1 cm) avec les capteurs T1 a T4 : la vue complete montre les conditions aux limites haute et basse, le zoom detaille les 50 premiers centimetres ou se trouvent les capteurs.


In [4]:
from Plot import plot_domain_schema

z_obs = [-0.105, -0.205, -0.305, -0.405, -1.005]   # capteurs T1 a T5, centres de maille a 10,5 / 20,5 / 30,5 / 40,5 / 100,5 cm
write_coordonnee_file(z_bottom, dz)  # ecrit E_coordonnee.dat, lu par plot_domain_schema
plot_domain_schema(1, None, z_top, z_bottom, z_obs=z_obs, zoom_depth=0.5)


/Users/ariviere/Programmes/ginette/src/src_python/Plot.py:831: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Calcul du nombre de Peclet (a faire avant de simuler)

Etapes : $K = k\rho g/\mu$, puis $q = -K\,\Delta h / L$, puis $\mathrm{Pe} = q\,C_w L/\lambda$. Completez le tableau pour $\Delta h = 1.5\cdot10^{-5}$, $1.5\cdot10^{-3}$ et $1.5\cdot10^{-1}$ m (les gradients utilises en section 5), avec les parametres par defaut : vous devez retrouver $|\mathrm{Pe}| \approx 0.01$, $1$ et $100$.

In [5]:
# Etape 1 : conductivite hydraulique
log_k_def, lam_def = -10.8, 1.0
K = 10 ** log_k_def * rho * g / mu
print(f'K = {K:.3e} m/s')

# Etape 2 et 3 : vitesse de Darcy puis Peclet, pour plusieurs differences de charge
for dh_test in [1.5e-5, 1.5e-3, 1.5e-1]:
    q_test = -K * dh_test / az
    Pe_test = q_test * Cw_vol * az / lam_def
    print(f'dh = {dh_test:+.2e} m  ->  q = {q_test:+.2e} m/s ({q_test*86400:+.3f} m/j)   Pe = {Pe_test:+.2f}')


K = 1.555e-04 m/s
dh = +1.50e-05 m  ->  q = -4.66e-10 m/s (-0.000 m/j)   Pe = -0.01
dh = +1.50e-03 m  ->  q = -4.66e-08 m/s (-0.004 m/j)   Pe = -0.98
dh = +1.50e-01 m  ->  q = -4.66e-06 m/s (-0.403 m/j)   Pe = -97.60


## 5. Trois niveaux de transport pour chaque sens d'ecoulement

Pour chaque sens (infiltrant, exfiltrant), trois cas de nombre de Peclet :
- **conduction** : $|\mathrm{Pe}| = 0.01$, le sens d'ecoulement existe mais la conduction domine ;
- **transition** : $|\mathrm{Pe}| \approx 1$ ;
- **advection** : $|\mathrm{Pe}| \approx 100$.

Avant de regarder les courbes, prevoyez : a 20,5 cm, quel cas se rapproche le plus de la surface, et lequel est le plus amorti ?

In [6]:
# Differences de charge calibrees sur le nombre de Peclet (|Pe| = |K*dh*Cw/lam| avec lam = 1)
K_cas = 10 ** -10.8 * rho * g / mu
dh_cond = 0.01 / (K_cas * Cw_vol)         # Pe = 0.01 : conduction dominante, sens fixe
dh_trans = 1.0 / (K_cas * Cw_vol)         # Pe = 1
dh_adv = 100.0 / (K_cas * Cw_vol)         # Pe = 100
scenarios = {
    'Conduction, infiltrant': dh_cond, 'Conduction, exfiltrant': -dh_cond,
    'Transition, infiltrant': dh_trans, 'Transition, exfiltrant': -dh_trans,
    'Advection, infiltrant': dh_adv, 'Advection, exfiltrant': -dh_adv,
}
resultats = {}
for nom, dh_s in scenarios.items():
    sim, q_s, Pe_s = simuler(dh=dh_s)
    resultats[nom] = sim
    print(f'{nom:28s} dh = {dh_s:+.6f} m   q = {q_s:+.2e} m/s   Pe = {Pe_s:+.3f}')


 simu done


Conduction, infiltrant       dh = +0.000015 m   q = -4.78e-10 m/s   Pe = -0.010


 simu done


Conduction, exfiltrant       dh = -0.000015 m   q = +4.78e-10 m/s   Pe = +0.010


 simu done


...

Transition, exfiltrant       dh = -0.001537 m   q = +4.78e-08 m/s   Pe = +1.000


 simu done


Advection, infiltrant        dh = +0.153686 m   q = -4.78e-06 m/s   Pe = -100.000


 simu done


Advection, exfiltrant        dh = -0.153686 m   q = +4.78e-06 m/s   Pe = +100.000


### Comprendre les sorties

Chaque simulation dure 40 jours : **10 jours de spin-up** (mise en route, le temps que la colonne oublie sa temperature initiale), puis **30 jours d'analyse**. Le spin-up est retire des resultats : `Time = 0` correspond au debut de l'analyse.

Chaque simulation renvoie trois types de resultats. Ils sont gardes en memoire dans le notebook : les fichiers `Sim_*.dat` du dossier `SYNTHETIC_CASES` sont ecrases a chaque run.

**1. Le tableau de temperatures** (`sim`, une ligne par pas de temps de 10 min)

| Colonne | Signification | Unite |
|---|---|---|
| `Time` | temps ecoule depuis la fin du spin-up (debut de l'analyse, `Time = 0`) | s |
| `dates` | date calendaire correspondante | - |
| `T10.5` a `T100.5` | temperature aux capteurs (centres de maille a 10,5 / 20,5 / 30,5 / 40,5 / 100,5 cm sous la surface) | °C |
| `Temp1` a `Temp3` | sorties brutes de Ginette, proches de T1 a T3 : redondantes, a ignorer | °C |

**2. Les vitesses d'eau** (`sim.attrs['flux_eau']`) : `Time`, `z` (profondeur de la maille), `q` (vitesse de Darcy, en m/s). Par convention, `q` est **positive vers le haut** : `q < 0` correspond a une infiltration (l'eau descend), `q > 0` a une exfiltration.

**3. Les flux de chaleur** (`sim.attrs['flux_chal']`) : `Conductive` $= -\lambda\,\partial T/\partial z$ (transport par diffusion), `Advective` $= \phi C_w\, q\, T$ (transport par l'eau), et `Total` leur somme, toutes en W/m².

Les profils de temperature sur toute la colonne (`sim.attrs['profils']`) donnent `Time`, `z` et `T` pour les jours de `PROFIL_JOURS`, comptes depuis le debut de l'analyse.

**Fichiers produits par Ginette** (dans `SYNTHETIC_CASES/`, ecrases a chaque simulation) :

| Fichier | Contenu | Taille | Utilisation dans le TD |
|---|---|---|---|
| `Sim_temperature_profil_t.dat` | `Time`, `z`, `T` : temperature de toutes les mailles, a chaque pas de temps | ~220 Mo | source de `T10.5` a `T100.5` et des profils |
| `Sim_velocity_profil_t.dat` | `Time`, `z`, `q` : vitesse de Darcy de toutes les mailles | ~220 Mo | `flux_eau` |
| `Sim_heat_flux_profil_t.dat` | `Time`, `z`, `Conductive`, `Advective`, `Total` : flux de chaleur (W/m²) | ~360 Mo | `flux_chal` |
| `Sim_temperature_maille1_t.dat` a `maille3` | `Time`, `T` : temperature de trois mailles fixes | ~140 ko | recopiees dans `Temp1` a `Temp3` (a ignorer) |
| `S_pression_charge_temperature.dat` | etat final (charge, pression, temperature) a la fin du calcul | ~70 ko | non lu par ce TD ; sert a reprendre un calcul |
| `E_*.dat` | fichiers d'**entree** ecrits par Python avant chaque run (conditions aux limites, geometrie, parametres) | - | a lire pour comprendre le probleme, pas a modifier a la main |

Les fichiers contiennent aussi le spin-up (les lignes avec `Time` < 10 jours) : elles sont retirees dans les resultats du notebook, mais restent dans les fichiers.

Les fichiers les plus lourds ne se lisent pas dans un editeur de texte : utilisez `pandas` comme dans les cellules du notebook.

Testez-le : la cellule suivante affiche ces tableaux pour le cas advectif infiltrant.


In [7]:
sim = resultats['Advection, infiltrant']
print('Colonnes de temperature :', list(sim.columns))
print('Grandeurs annexes       :', list(sim.attrs.keys()))
print()
print('Temperatures (5 premieres lignes) :')
display(sim[['Time', 'dates', 'T10.5', 'T20.5', 'T100.5']].head())
print('Vitesses de Darcy (5 premieres lignes) :')
display(sim.attrs['flux_eau'].head())
print('Flux de chaleur (5 premieres lignes) :')
display(sim.attrs['flux_chal'].head())


Colonnes de temperature : ['Time', 'Temp1', 'Temp2', 'Temp3', 'dates', 'T10.5', 'T20.5', 'T30.5', 'T40.5', 'T100.5']
Grandeurs annexes       : ['flux_eau', 'flux_chal', 'profils']

Temperatures (5 premieres lignes) :


,Time,dates,T10.5,T20.5,T100.5
0,0.0,2022-05-01 14:00:00,14.104478,14.759778,16.974647
1,600.0,2022-05-01 14:10:00,14.186753,14.726020,16.968946
2,1200.0,2022-05-01 14:20:00,14.274384,14.696590,16.963304
3,1800.0,2022-05-01 14:30:00,14.367203,14.671545,16.957731
4,2400.0,2022-05-01 14:40:00,14.465033,14.650932,16.952239


Vitesses de Darcy (5 premieres lignes) :


,Time,z,q
0,0.0,-0.005,-0.000005
1,0.0,-0.015,-0.000005
2,0.0,-0.025,-0.000005
3,0.0,-0.035,-0.000005
4,0.0,-0.045,-0.000005


Flux de chaleur (5 premieres lignes) :


,Time,z,Conductive,Advective,Total
0,0.0,-0.005,-34.539082,-329.672750,-364.211832
1,0.0,-0.015,-31.453378,-321.466734,-352.920111
2,0.0,-0.025,-28.289575,-313.993838,-342.283413
3,0.0,-0.035,-25.082535,-307.272620,-332.355155
4,0.0,-0.045,-21.865100,-301.313350,-323.178450


### Utiliser les fichiers de profils directement

Les trois fichiers de profils contiennent la solution complete du modele : **une ligne par maille et par pas de temps**. Avec `dz` = 1 cm sur 5 m et un pas de 10 min pendant 40 jours (10 jours de spin-up et 30 jours d'analyse), cela fait 500 x 5760 = 2 880 000 lignes par fichier. La cellule suivante retire le spin-up (`Time` < 10 jours) et remet le temps a zero : il reste 4321 pas de temps (de 0 a 30 jours, bornes comprises), soit 2 160 500 lignes. Ils restent donc utilisables directement, a condition de les lire une seule fois et de les organiser en tableau (temps x profondeur).

| Fichier | Colonnes | Unites | Convention |
|---|---|---|---|
| `Sim_temperature_profil_t.dat` | `Time`, `z`, `T` | s, m, °C | `z` negatif sous la surface |
| `Sim_velocity_profil_t.dat` | `Time`, `z`, `q` | s, m, m/s | `q` positif vers le haut (exfiltration) |
| `Sim_heat_flux_profil_t.dat` | `Time`, `z`, `Conductive`, `Advective`, `Total` | s, m, W/m² | `Conductive` = $-\lambda\,\partial T/\partial z$ ; `Advective` = $\phi C_w\, q\, T$ |

Les centres de mailles sont a $z = -0{,}005$, $-0{,}015$, $-0{,}025$ m, etc. ; la cote $-0{,}205$ m (20,5 cm) est donc exactement une maille.

**Attention** : ces fichiers correspondent a la **derniere simulation lancee** (ici le cas advectif exfiltrant), car ils sont ecrases a chaque run. Pour comparer des cas, gardez les resultats en memoire, comme dans `resultats`.

La cellule suivante lit les trois fichiers et trace le champ de temperature (temps x profondeur), la vitesse de Darcy et les flux a 20,5 cm.


In [8]:
# Lecture des fichiers de profils (dossier SYNTHETIC_CASES, derniere simulation lancee)
prof_T = pd.read_csv(synth_dir / 'Sim_temperature_profil_t.dat', sep=r'\s+', header=None,
                     names=['Time', 'z', 'T'])
prof_q = pd.read_csv(synth_dir / 'Sim_velocity_profil_t.dat', sep=r'\s+', header=None,
                     names=['Time', 'z', 'q'])
prof_F = pd.read_csv(synth_dir / 'Sim_heat_flux_profil_t.dat', sep=r'\s+', header=None,
                     names=['Time', 'z', 'Conductive', 'Advective', 'Total'])
# meme traitement que dans sim : on retire le spin-up et on remet le temps a zero (retirer_spin_up)
prof_T, prof_q, prof_F = [retirer_spin_up(d) for d in (prof_T, prof_q, prof_F)]
print(prof_T.shape, prof_q.shape, prof_F.shape)

# 1. Champ de temperature : temps (jours) en abscisse, profondeur en ordonnee
T_mat = prof_T.pivot(index='z', columns='Time', values='T')
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
im = axes[0].imshow(T_mat.values, aspect='auto', origin='lower', cmap='coolwarm',
                    extent=[T_mat.columns.min() / 86400, T_mat.columns.max() / 86400,
                            T_mat.index.min(), T_mat.index.max()])
fig.colorbar(im, ax=axes[0], label='Temperature [°C]')
axes[0].set_xlabel('Temps [jours]'); axes[0].set_ylabel('z [m]'); axes[0].set_title('Temperature')

# 2. Vitesse de Darcy a 20,5 cm
q_20 = prof_q[np.isclose(prof_q['z'], -0.205)]
axes[1].plot(q_20['Time'] / 86400, q_20['q'])
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_title('Vitesse de Darcy a 20,5 cm'); axes[1].set_xlabel('Temps [jours]'); axes[1].set_ylabel('q [m/s]')

# 3. Flux de chaleur a 20,5 cm
F_20 = prof_F[np.isclose(prof_F['z'], -0.205)]
for col in ['Conductive', 'Advective', 'Total']:
    axes[2].plot(F_20['Time'] / 86400, F_20[col], label=col)
axes[2].set_title('Flux de chaleur a 20,5 cm'); axes[2].set_xlabel('Temps [jours]'); axes[2].set_ylabel('W/m²')
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()


(2160500, 3) (2160500, 3) (2160500, 5)


/var/folders/4y/y873lp5j2z12snsd4wwbzy1c0000gp/T/ipykernel_28308/1386208257.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, sens in zip(axes, ['infiltrant', 'exfiltrant']):
    for regime in ['Conduction', 'Transition', 'Advection']:
        ax.plot(resultats[f'{regime}, {sens}']['Time'] / 86400,
                resultats[f'{regime}, {sens}']['T20.5'], label=regime)
    ax.set_title(f'Capteur a 20,5 cm - {sens}')
    ax.set_xlabel('Temps [jours]'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel('Temperature [°C]')
plt.tight_layout(); plt.show()


/var/folders/4y/y873lp5j2z12snsd4wwbzy1c0000gp/T/ipykernel_28308/3782282814.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Sorties Ginette : temperature, flux d'eau et flux de chaleur

- **Temperature** : evolution a 20,5 cm (capteur T2).
- **Flux d'eau** : vitesse de Darcy $q$ a 20,5 cm [m/s], positive vers le haut. C'est le debit specifique qui traverse l'interface.
- **Flux de chaleur** : flux conductif $-\lambda\,\partial T/\partial z$ et flux advectif $\phi C_w q T$, en W/m². Leur somme est le flux total. Observez quelle part domine dans chaque regime.

In [10]:
def serie_profondeur(sim, df, colonne, z_cible=-0.205):
    z_u = df['z'].unique()
    z_proche = z_u[np.argmin(np.abs(z_u - z_cible))]
    d = df[np.isclose(df['z'], z_proche)]
    return d['Time'].values / 86400, d[colonne].values

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for ligne, sens in enumerate(['infiltrant', 'exfiltrant']):
    for regime in ['Conduction', 'Transition', 'Advection']:
        sim = resultats[f'{regime}, {sens}']
        t, T = serie_profondeur(sim, sim.rename(columns={'Time':'Time'}).assign(z=0.0) if False else None, None) if False else (sim['Time'].values/86400, sim['T20.5'].values)
        axes[ligne, 0].plot(t, T, label=regime)
        t, q = serie_profondeur(sim, sim.attrs['flux_eau'], 'q')
        axes[ligne, 1].plot(t, q, label=regime)
        t, adv = serie_profondeur(sim, sim.attrs['flux_chal'], 'Advective')
        t, cond = serie_profondeur(sim, sim.attrs['flux_chal'], 'Conductive')
        axes[ligne, 2].plot(t, cond + adv, label=f'{regime} (total)')
    axes[ligne, 0].set_ylabel(f'{sens}\nTemperature [°C]')
    axes[ligne, 1].set_ylabel('Flux d\'eau q [m/s]')
    axes[ligne, 2].set_ylabel('Flux de chaleur [W/m²]')
for ax, titre in zip(axes[0], ['Temperature a 20,5 cm', 'Vitesse de Darcy a 20,5 cm', 'Flux de chaleur total a 20,5 cm']):
    ax.set_title(titre)
for ax in axes[-1]:
    ax.set_xlabel('Temps [jours]')
for ax in axes.ravel():
    ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


/var/folders/4y/y873lp5j2z12snsd4wwbzy1c0000gp/T/ipykernel_28308/3166910764.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


### Lire les figures

- **Conduction** : l'onde journaliere (amplitude 5 °C en surface) est fortement attenuee et retardee par diffusion : a 20,5 cm il ne reste que quelques dixiemes de degre. Le sens du flux ne change rien de visible : c'est le transport par diffusion, qui est symetrique.
- **Transition** : le sens du flux commence a se voir. Infiltration et exfiltration ne donnent pas la meme courbe, car l'eau transporte la chaleur dans un sens donne.
- **Advection** : le signal de surface arrive presque sans attenuation en infiltration (l'eau froide ou chaude descend vite), alors qu'en exfiltration la nappe, plus stable a 10 °C, impose son niveau et attenue la surface.

Question : pour l'infiltration, a quelle vitesse le signal de surface voyage-t-il ? Estimez-la sur la figure, puis comparez a $|q|/n$ (vitesse reelle de l'eau).

In [11]:
# Profils verticaux de temperature aux dates PROFIL_JOURS, pour chaque regime et sens d'ecoulement
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey=True)
couleurs = plt.cm.viridis(np.linspace(0, 1, len(PROFIL_JOURS)))
for ligne, sens in enumerate(['infiltrant', 'exfiltrant']):
    for colonne, regime in enumerate(['Conduction', 'Transition', 'Advection']):
        ax = axes[ligne, colonne]
        pr = resultats[f'{regime}, {sens}'].attrs['profils']
        for jour, couleur in zip(PROFIL_JOURS, couleurs):
            p_j = pr[np.isclose(pr['Time'], jour * 86400)].sort_values('z', ascending=False)
            ax.plot(p_j['T'].values, p_j['z'].values, '-', color=couleur, label=f'{jour} j')
        ax.set_title(f'{regime} - {sens}')
        ax.grid(alpha=0.3)
    axes[ligne, 0].set_ylabel('Profondeur [m]')
for ax in axes[-1]:
    ax.set_xlabel('Temperature [°C]')
axes[0, 0].legend(fontsize=8, title='Date')
plt.tight_layout(); plt.show()


/var/folders/4y/y873lp5j2z12snsd4wwbzy1c0000gp/T/ipykernel_28308/1497963671.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Sur le profil, observez : la temperature de la surface penetre-t-elle plus loin en infiltration ? Pourquoi l'exfiltration rapproche-t-elle les profils de la temperature de la base ?

### Synthese

| Regime | |Pe| | Ce qu'il faut retenir |
|---|---|---|
| Conduction | 0.01 | Diffusion seule : retard et attenuation, sens du flux invisible |
| Transition | 1 | Le sens du flux commence a modifier le signal |
| Advection | 100 | L'eau transporte la chaleur : asymetrie infiltration/exfiltration |

Pour aller plus loin : refaites l'exercice avec une porosite $n = 0.2$ puis $n = 0.4$ dans le regime de transition, et expliquez le resultat a l'aide de $C_v$ et de $\lambda$.

## 6. Effet des parametres (regime infiltrant)

Faites varier un parametre a la fois. Le signal a 40,5 cm s'amortit-il et se decale-t-il plus ou moins ? Pourquoi ?

Pistes :
- porosite $\phi$ : agit sur la capacite $C_v$ et sur la conductivite equivalente ;
- conductivite $\lambda$ : diffusion thermique pure ;
- permeabilite $k$ : vitesse de Darcy, donc nombre de Peclet.


In [12]:
variations = {'n = 0.2': dict(dh=0.02, n=0.2), 'n = 0.4': dict(dh=0.02, n=0.4),
              'lam = 0.5': dict(dh=0.02, lam=0.5), 'lam = 2.0': dict(dh=0.02, lam=2.0),
              'k = 1e-11': dict(dh=0.02, log_k=-11.0), 'k = 1e-10': dict(dh=0.02, log_k=-10.0)}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for nom, p in variations.items():
    sim, q, Pe = simuler(**p)
    axes[0].plot(sim['Time'] / 86400, sim['T20.5'], label=f"{nom} (Pe={Pe:+.2f})")
    axes[1].plot(sim['Time'] / 86400, sim['T10.5'], label=nom)
    axes[2].plot(sim['Time'] / 86400, sim['T100.5'], label=nom)
for ax, titre in zip(axes, ['Capteur a 20,5 cm', 'Capteur a 10,5 cm', 'Capteur a 100,5 cm']):
    ax.set_title(titre); ax.set_xlabel('Temps [jours]'); ax.grid(alpha=0.3)
axes[0].set_ylabel('Temperature [°C]'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


 simu done


 simu done


 simu done


 simu done


 simu done


 simu done


/var/folders/4y/y873lp5j2z12snsd4wwbzy1c0000gp/T/ipykernel_28308/1484229456.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7. Questions

1. Pour un regime infiltrant, pourquoi la temperature a 40,5 cm suit-elle davantage la surface que dans le cas conductif ?
2. Pour l'exfiltration, quel est le sens de propagation du signal ? Pourquoi la base (10 °C) pese-t-elle davantage ?
3. Changer `dz` : que devient le nombre de mailles, et la position des capteurs ? Pourquoi le resultat change-t-il peu si `dz` reste petit devant la longueur caracteristique du signal ?
4. Pour quelle valeur de $|q|$ le nombre de Peclet atteint-il 1 avec les parametres par defaut ?
